# 3D Toric-Code NQS — hyperparameter sweep on Colab

Runs the **same** `Three_TC.train` pipeline as the NERSC GPU job array, but on
Colab's single GPU: the Slurm array just becomes a sequential loop over the same
grid.

**Just run the cells top to bottom.** To explore *different values*, edit the grid
and the fixed args in the **“Configure the sweep”** cell — that's the only place
you need to touch.

Each run does dense-QGT Stochastic Reconfiguration on `ToricCNN_full` and prints,
every step, `E ± ΔE`, the spread `√Var`, and the figure of merit
**`delta = |E − E_exact| / |E_exact|`** (also logged live to Weights & Biases).

> First: **Runtime → Change runtime type → GPU**, then run cell 1 to confirm.

In [ ]:
# 1. Confirm a GPU is attached (Runtime -> Change runtime type -> GPU).
!nvidia-smi -L || echo "NO GPU -- switch the runtime to GPU before continuing."

In [ ]:
# 2. Clone (or update) the repo into the Colab VM. Idempotent: safe to re-run.
import os
REPO_URL = "https://github.com/SanzharBissenali/ThreeD_TC.git"
REPO_DIR = "/content/ThreeD_TC"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git pull --ff-only || true
print("cwd:", os.getcwd())

In [ ]:
# 3. Install the NQS stack, pinned to match the NERSC env (parity of results).
#    Colab may print dependency-conflict warnings for unrelated preinstalled
#    packages (tensorflow, etc.) -- those are harmless for this pipeline.
!pip install -q jax==0.5.2 jaxlib==0.5.1 \
    jax-cuda12-plugin==0.5.1 jax-cuda12-pjrt==0.5.1 \
    netket==3.16.1.post1 flax==0.10.4 optax numba wandb tqdm

In [ ]:
# 4. GATE: make sure JAX actually sees the GPU (expect platform 'gpu').
#    If this fails: re-run cell 3, then Runtime -> Restart session, then re-run
#    from cell 2. (x64 is already enabled inside Three_TC/train.py.)
import os
# Allocate GPU memory ON DEMAND instead of grabbing ~75% up front. Set BEFORE
# importing jax so the kernel AND every sweep subprocess (they inherit os.environ)
# play nice; otherwise back-to-back runs in the loop fight over preallocated VRAM
# and the 2nd one OOMs right after wandb.init (run appears, no steps). See cell 7.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
import jax
print("devices:", jax.devices(), "| backend:", jax.default_backend())
assert jax.default_backend() == "gpu", "JAX is not on the GPU -- see the note above."

## Weights & Biases

`WANDB = True` logs every step's `delta` / `energy` / `energy_abs_err` to a shared
group so all configs plot side by side (a parallel-coordinates plot over `delta` is
the intended view). Use a group name **distinct from your NERSC group** so the two
don't mix — same project, so they're still visible together.

Set `WANDB = False` to skip it entirely and just watch the per-step stdout.

In [ ]:
# 5. W&B auth. Set WANDB = False to skip and rely on stdout only.
WANDB = True
if WANDB:
    import wandb
    wandb.login()   # paste your API key when prompted

## Configure the sweep  ← edit here

`GRID` is the Cartesian product of the four axis lists (this mirrors
`scripts/sweep_params.py`, the grid NERSC uses). Edit the lists for **different
values**; edit the fixed args for the physical point and run length.

Axes: `(initial lr, final lr)` cosine decay · SR/QGT `diag_shift` · pre-Wilson
capacity `(n_noninv, noninv_channels)` · post-Wilson `inv_hidden` widths.

In [ ]:
import itertools

# === EDIT: the hyperparameter grid ("different values") =====================
DT_LRMIN    = [(0.02, 0.002), (0.01, 0.001)]   # (initial lr, final lr)
DIAG_SHIFT  = [1e-4, 1e-3, 1e-2]               # SR / QGT regularization
PRE_WILSON  = [(1, 1), (2, 1), (1, 4)]         # (n_noninv, noninv_channels)
POST_WILSON = [(8,), (16, 16)]                 # inv_hidden widths

# === EDIT: fixed args shared by every run ==================================
HZ_PRESET = "easy"     # ED reference point: "hard" | "mid" | "easy"
N_ITER    = 250
N_SAMPLES = 16384
N_CHAINS  = 1024
# ===========================================================================

def flags(combo):
    (dt, lr_min), ds, (n_noninv, nc), inv_hidden = combo
    return (f"--dt {dt} --lr_min {lr_min} --diag_shift {ds} "
            f"--n_noninv {n_noninv} --noninv_channels {nc} "
            f"--inv_hidden {' '.join(map(str, inv_hidden))}")

GRID = list(itertools.product(DT_LRMIN, DIAG_SHIFT, PRE_WILSON, POST_WILSON))
WANDB_GROUP = f"colab-sweep-{HZ_PRESET}"
print(f"{len(GRID)} configs"
      + (f"; wandb group '{WANDB_GROUP}'" if WANDB else "; wandb OFF"))
for i, c in enumerate(GRID):
    print(f"  cfg{i:2d}: {flags(c)}")

## Sanity check — one short run

A quick `cfg0` at reduced iters/samples to confirm training works end-to-end before
committing to the full sweep. You should see the per-step `delta = ...` stream.

In [ ]:
# 6. One short smoke run (cfg0).
i = 0
cfg_flags = flags(GRID[i])
wb = f"--wandb_group {WANDB_GROUP}" if WANDB else "--no_wandb"
!python -m Three_TC.train --L 2 --bc PBC --model bosonic --arch ToricCNN_full \
  --hx 0.2 --hz_preset {HZ_PRESET} --n_iter 50 --n_samples 4096 --n_chains 256 \
  {wb} --name cfg0_smoke {cfg_flags}

## Run the sweep

Sequential — Colab has one GPU. Set `RUN_INDICES` to a subset like `range(4)` for a
quick look, or `range(len(GRID))` for the whole grid. Each run prints live and
writes `outputs/cfg<i>.{json,mpack}`.

> Colab sessions disconnect when idle (~90 min) and cap around 12 h; the full
> grid runs back-to-back, so for exploration prefer a subset or smaller
> `N_ITER`/`N_SAMPLES`.

In [ ]:
# 7. The sweep loop (Colab analogue of Slurm --array=0-N). Each config runs as a
#    FRESH subprocess so JAX / GPU / wandb state can't leak between runs.
import os, sys, time, subprocess

RUN_INDICES = range(len(GRID))   # e.g. range(4) for a quick subset

# On-demand GPU allocator (matches cell 4) so sequential runs don't fight over
# VRAM; unbuffered so each run's output + any traceback show up live.
env = {**os.environ,
       "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
       "XLA_PYTHON_CLIENT_ALLOCATOR": "platform",
       "PYTHONUNBUFFERED": "1"}

failures = []
for i in RUN_INDICES:
    wb = ["--wandb_group", WANDB_GROUP] if WANDB else ["--no_wandb"]
    cmd = [sys.executable, "-u", "-m", "Three_TC.train",
           "--L", "2", "--bc", "PBC", "--model", "bosonic",
           "--arch", "ToricCNN_full", "--hx", "0.2", "--hz_preset", HZ_PRESET,
           "--n_iter", str(N_ITER), "--n_samples", str(N_SAMPLES),
           "--n_chains", str(N_CHAINS), "--name", f"cfg{i}",
           *wb, *flags(GRID[i]).split()]
    print(f"\n===== config {i}/{len(GRID)-1}: {flags(GRID[i])} =====", flush=True)
    r = subprocess.run(cmd, env=env)          # exit code checked below, not swallowed
    if r.returncode != 0:
        print(f"!! cfg{i} FAILED (exit {r.returncode}) -- traceback above", flush=True)
        failures.append(i)
    time.sleep(2)                             # let the driver reclaim VRAM first

print(f"\nsweep done. failures: {failures}" if failures else "\nsweep done.")

## Notes

**Different values — three knobs:**
- *Different grid* → edit the four axis lists in the configure cell.
- *Different physical point* → change `HZ_PRESET` (`hard`/`mid`/`easy`). For a
  custom `h_z`, replace `--hz_preset ...` with `--hz <value> --exact_E0 <E>` in the
  commands (you need an ED reference `E` for the `delta` FOM).
- *One-off override* → append any flag after `flags(GRID[i])`; argparse takes the
  last value (e.g. add `--n_iter 400` to deepen a single run).

**Outputs.** Each run writes `outputs/cfg<i>.json` (config + curve + final
observables) and `outputs/cfg<i>.mpack` (weights). The Colab VM is ephemeral —
download keepers with the cell below, or rely on W&B.

In [ ]:
# Optional: zip + download the run JSONs/weights before the VM recycles.
from google.colab import files
!cd outputs && zip -qr /content/colab_sweep_outputs.zip . && echo "zipped outputs/"
files.download("/content/colab_sweep_outputs.zip")